# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** **Line chart** of an index (1970 = 100) per continent.

I used a line chart because the question is about how things change over time, and lines are good for that. I made every continent start at 100 in 1970, so it’s fair and the big ones don’t look faster just because they’re big. Africa is orange because it grew the fastest. The rest are grey so they don’t distract you. I put the names at the end of the lines so you don’t have to look at a legend.

In [ ]:
# Q1
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100

colors = {c: '#BBBBBB' for c in cont['Continent'].unique()}
colors['Africa'] = '#E69F00'

fig = px.line(cont, x='Year', y='Index', color='Continent',
              color_discrete_map=colors, markers=True,
              template='plotly_white',
              title="<b>Africa's population has nearly quadrupled since 1970</b>"
                    "<br><sup>Population indexed to 1970 = 100. Every other continent grew 1.9–2.3×; Europe only 1.13×</sup>")

fig.update_traces(line_width=2, marker_size=5, hovertemplate='%{y:.0f}')
fig.update_traces(selector=dict(name='Africa'), line_width=4, marker_size=7)

# Direct labels at the 2022 end of each line instead of a legend
# (nudge labels apart where lines end close together: Asia, S. America, Oceania)
end = cont[cont['Year'] == 2022].sort_values('Index').reset_index(drop=True)
end['LabelY'] = end['Index']
for i in range(1, len(end)):
    end.loc[i, 'LabelY'] = max(end.loc[i, 'Index'], end.loc[i - 1, 'LabelY'] + 14)
for _, r in end.iterrows():
    is_af = r['Continent'] == 'Africa'
    label = f"{r['Continent']} {r['Index'] / 100:.1f}×"
    fig.add_annotation(x=2022, y=r['LabelY'], xanchor='left', xshift=8, showarrow=False,
                       text=f'<b>{label}</b>' if is_af else label,
                       font=dict(color='#E69F00' if is_af else '#777777', size=12))

fig.add_hline(y=100, line_dash='dot', line_color='#999999')

fig.update_layout(showlegend=False, width=850, height=500,
                  margin=dict(l=60, r=170, t=90, b=50))
fig.update_xaxes(title=None, showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2022])
fig.update_yaxes(title='Index (1970 = 100)', gridcolor='#EEEEEE', zeroline=False)
fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** **Sorted horizontal bar chart**.

I used a bar chart sorted from biggest to smallest, so you can see who added the most people. The bars go sideways so the country names fit. The African countries are orange and the others are grey, so you can count them fast (there are 4). I wrote the numbers on the bars, so I hid the bottom axis.

In [ ]:
# Q2
df['Added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Added').copy()
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
top15['Label'] = (top15['Added'] / 1e6).round().astype(int).astype(str) + 'M'
n_africa = (top15['Continent'] == 'Africa').sum()

fig = px.bar(top15.sort_values('Added'), x='Added', y='Country', orientation='h',
             color='Group', text='Label',
             color_discrete_map={'Africa': '#E69F00', 'Other continents': '#BBBBBB'},
             template='plotly_white',
             title=f"<b>{n_africa} of the 15 countries that added the most people since 1970 are in "
                   f"<span style='color:#E69F00'>Africa</span></b>"
                   "<br><sup>People added between 1970 and 2022. India and China alone added 1.46 billion</sup>")

fig.update_traces(textposition='outside', cliponaxis=False, hovertemplate='%{y}: %{x:,.0f}')
fig.update_layout(showlegend=False, width=850, height=550, bargap=0.25,
                  margin=dict(l=110, r=60, t=90, b=30))
fig.update_xaxes(visible=False)
fig.update_yaxes(title=None, categoryorder='total ascending')
fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** **Scatter plot** on **log–log axes**, coloured by continent.

I used a scatter plot because I wanted to see whether two numbers go together: size of the country and number of people. Some countries are super tiny and some are huge, so I used a log scale, otherwise all the dots would be squished in a corner. The colours show the continents. The grey dashed line shows the normal pattern: bigger countries usually have more people. I wrote names on the weird ones, like Macau and Singapore (tiny but full of people) and Greenland (huge but almost empty).

In [ ]:
# Q3
cont_colors = {'Africa': '#E69F00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
               'North America': '#56B4E9', 'Oceania': '#000000', 'South America': '#6A3D9A'}

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent',
                 hover_name='Country', log_x=True, log_y=True,
                 color_discrete_map=cont_colors,
                 category_orders={'Continent': list(cont_colors)},
                 template='plotly_white',
                 title="<b>Bigger countries generally have more people — but small Asian hubs and "
                       "empty northern lands break the pattern</b>"
                       "<br><sup>Land area vs 2022 population, log scales. Dashed line = overall trend "
                       "(r = 0.87 on log scales)</sup>")
fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='white')))

# Overall trend line (least squares on log10 values)
lx, ly = np.log10(df['Area']), np.log10(df['2022 Population'])
b, a = np.polyfit(lx, ly, 1)
xs = np.logspace(lx.min(), lx.max(), 50)
fig.add_scatter(x=xs, y=10 ** (a + b * np.log10(xs)), mode='lines', name='Trend',
                line=dict(color='#999999', dash='dash', width=1.5),
                hoverinfo='skip', showlegend=False)

# Label the countries furthest from the trend
outliers = {'Macau': (0, -25), 'Singapore': (-45, -20), 'Hong Kong': (35, -25),
            'Bangladesh': (-55, -20), 'India': (-35, -15),
            'Greenland': (0, 25), 'Falkland Islands': (0, 25), 'Iceland': (30, 25)}
for name, (ax, ay) in outliers.items():
    r = df[df['Country'] == name].iloc[0]
    fig.add_annotation(x=np.log10(r['Area']), y=np.log10(r['2022 Population']),
                       text=name, ax=ax, ay=ay, showarrow=True, arrowhead=0,
                       arrowcolor='#777777', font=dict(size=11, color='#333333'))

fig.update_layout(width=900, height=600, legend_title_text=None,
                  legend=dict(orientation='h', yanchor='bottom', y=1.0, x=0),
                  margin=dict(l=70, r=30, t=120, b=60))
fig.update_xaxes(title='Land area (km², log scale)', showgrid=False)
fig.update_yaxes(title='Population 2022 (log scale)', gridcolor='#EEEEEE')
fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** **Box plot** per continent, on a log scale, with every country shown as a point.

I used a box plot because it shows the middle and also how spread out the countries are, not just the average. I also showed every country as a dot. Some places are super crowded (Macau) and some are almost empty (Greenland), so I used a log scale. I sorted the continents from least crowded to most crowded. All the boxes are the same colour because colour wouldn’t tell you anything extra here.

In [ ]:
# Q4
order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()

fig = px.box(df, x='Continent', y='Density', points='all', hover_name='Country',
             log_y=True, category_orders={'Continent': order},
             color_discrete_sequence=['#0072B2'],
             template='plotly_white',
             title="<b>Density varies far more within continents than between them</b>"
                   "<br><sup>People per km² in 2022, log scale; each dot is a country. Medians range from 18 "
                   "(South America) to 221 (North America, driven by small Caribbean islands)</sup>")
fig.update_traces(marker=dict(size=5, opacity=0.6), line_width=1.5,
                  fillcolor='rgba(0,114,178,0.15)', jitter=0.4, pointpos=0)

for name in ['Macau', 'Monaco', 'Greenland']:
    r = df[df['Country'] == name].iloc[0]
    fig.add_annotation(x=r['Continent'], y=np.log10(r['Density']), text=name,
                       xshift=12, xanchor='left', showarrow=False,
                       font=dict(size=11, color='#555555'))

fig.update_layout(width=850, height=550, showlegend=False,
                  margin=dict(l=70, r=30, t=100, b=50))
fig.update_xaxes(title=None)
fig.update_yaxes(title='People per km² (log scale)', gridcolor='#EEEEEE')
fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** **Choropleth map** with a **diverging colour scale** centred on 0 %.

I used a map because the question asks where in the world it happened. Orange means the country got smaller and purple means it got bigger. White means almost no change. I didn’t use red and green because some people can’t tell them apart. I stopped the colours at ±50% so Qatar (which grew a crazy amount) doesn’t make everything else look the same.

In [ ]:
# Q5
df['Change 2000–2022 (%)'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100
n_shrank = (df['Change 2000–2022 (%)'] < 0).sum()
n_shrank_eu = ((df['Change 2000–2022 (%)'] < 0) & (df['Continent'] == 'Europe')).sum()

fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3',
                    color='Change 2000–2022 (%)', hover_name='Country',
                    color_continuous_scale='PuOr', range_color=[-50, 50],
                    color_continuous_midpoint=0,
                    projection='natural earth',
                    template='plotly_white',
                    title=f"<b>{n_shrank} territories shrank between 2000 and 2022, {n_shrank_eu} of them in Europe; "
                          "Africa and the Gulf grew fastest</b>"
                          "<br><sup>Population change 2000–2022 (%). Orange = shrank, purple = grew; "
                          "colour capped at ±50 %</sup>")

fig.update_traces(marker_line_color='white', marker_line_width=0.3,
                  hovertemplate='<b>%{hovertext}</b><br>%{z:+.1f}%<extra></extra>')
fig.update_geos(showframe=False, showcoastlines=False, showcountries=False,
                showland=True, landcolor='#EEEEEE', bgcolor='rgba(0,0,0,0)')
fig.update_layout(width=950, height=550, margin=dict(l=10, r=10, t=90, b=10),
                  coloraxis_colorbar=dict(title='% change', ticksuffix='%', len=0.6,
                                          tickvals=[-50, -25, 0, 25, 50],
                                          ticktext=['≤ −50%', '−25%', '0%', '+25%', '≥ +50%']))
fig.show()


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** **Treemap** (World → Continent → Country).

I used a treemap because it shows the whole world split into continents, and then the continents split into countries, all in one picture. Bigger box = more people. Asia is blue because it’s the biggest part (59%), and the other continents are grey. I wrote the percent on each box so you don’t need a legend.

In [ ]:
# Q6
tm_colors = {'Asia': '#0072B2', 'Africa': '#BBBBBB', 'Europe': '#C8C8C8',
             'North America': '#BBBBBB', 'South America': '#C8C8C8', 'Oceania': '#BBBBBB',
             '(?)': '#FFFFFF'}

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'],
                 values='2022 Population', color='Continent',
                 color_discrete_map=tm_colors,
                 template='plotly_white',
                 title="<b><span style='color:#0072B2'>Asia</span> is home to 59% of the world's 8 billion people "
                       "— India and China alone hold over a third</b>"
                       "<br><sup>2022 population by continent and country; tile area ∝ population</sup>")

fig.update_traces(textinfo='label+percent root', root_color='white',
                  marker=dict(line=dict(color='white', width=1)),
                  hovertemplate='<b>%{label}</b><br>%{value:,.0f} people<br>%{percentRoot:.1%} of world<extra></extra>')
fig.update_layout(width=950, height=600, margin=dict(l=10, r=10, t=80, b=10))
fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** **100 % stacked bar chart**, one bar per year.

used a 100% stacked bar chart because the question is about shares, and every bar adds up to 100%. Africa is orange and Europe is blue because those two changed the most: Africa went up and Europe went down. Asia and the rest are grey because they stayed about the same. Africa is at the bottom so it’s easy to see it getting bigger. I wrote the names next to the last bar instead of using a legend.

In [12]:
# Q7
groups = ['Africa', 'Asia', 'Europe']
shares = df_long.copy()
shares['Group'] = shares['Continent'].where(shares['Continent'].isin(groups), 'Rest of the world')
shares = shares.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
shares['Share'] = shares['Population'] / shares.groupby('Year')['Population'].transform('sum') * 100

# Years are unevenly spaced, so give each bar a plain position 0..7 and label it with the year
years = sorted(shares['Year'].unique())
shares['Pos'] = shares['Year'].map({y: i for i, y in enumerate(years)})

stack_order = ['Africa', 'Europe', 'Rest of the world', 'Asia']
share_colors = {'Africa': '#E69F00', 'Europe': '#0072B2',
                'Rest of the world': '#DDDDDD', 'Asia': '#BBBBBB'}

fig = px.bar(shares, x='Pos', y='Share', color='Group', text='Share',
             category_orders={'Group': stack_order},
             color_discrete_map=share_colors,
             template='plotly_white',
             title="<b>Africa's share of the world nearly doubled since 1970, while Europe's halved</b>"
                   "<br><sup>Share of world population (%). Asia and the rest of the world stayed roughly flat</sup>")

fig.update_traces(texttemplate='%{text:.0f}%', textposition='inside')

# Group names next to the 2022 bar, so no legend is needed
last = shares[shares['Year'] == 2022].set_index('Group')['Share'].reindex(stack_order)
mid = last.cumsum() - last / 2
for g in stack_order:
    fig.add_annotation(x=len(years) - 1, y=float(mid[g]), text='<b>' + g + '</b>',
                       xanchor='left', xshift=40, showarrow=False,
                       font=dict(color=share_colors[g] if g in ['Africa', 'Europe'] else '#555555'))

fig.update_layout(barmode='stack', showlegend=False, bargap=0.25,
                  width=900, height=520, margin=dict(l=30, r=160, t=90, b=40))
fig.update_xaxes(title=None, tickvals=list(range(len(years))), ticktext=[str(y) for y in years])
fig.update_yaxes(visible=False, range=[0, 100])
fig.show()